# **03 - SelfCheckNLI**


In [ ]:
# Restart after running this cell
!pip install -q "transformers==4.36.2" selfcheckgpt sentencepiece "accelerate>=0.26,<1.0" "bitsandbytes>=0.43.0"
!python -m spacy download en_core_web_sm -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to 

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/selfcheckgpt_replication_results"
except ImportError:                      # running outside Colab
    RESULTS_DIR = "./selfcheckgpt_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["SC_RESULTS_DIR"] = RESULTS_DIR
if RESULTS_DIR not in sys.path: sys.path.insert(0, RESULTS_DIR)
try:
    from sc_common import *            # shared config, dataset, metrics, cache helpers 
except ModuleNotFoundError:
    raise FileNotFoundError(f"Upload sc_common.py into {RESULTS_DIR} first.")
describe_device(); ram("after setup")

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/evaluation-00000-of-00001-e91191b8f(…):   0%|          | 0.00/2.56M [00:00<?, ?B/s]

Generating evaluation split:   0%|          | 0/238 [00:00<?, ? examples/s]

Using device: cuda
Tesla T4 - 14.6 GB
Results / caches -> /content/drive/MyDrive/selfcheckgpt_replication_results
[RAM after setup] this process 0.8 GB | system used 1.7 / 12.7 GB


In [2]:
from selfcheckgpt.modeling_selfcheck import SelfCheckNLI
selfcheck_nli = SelfCheckNLI(device=device)

@torch.no_grad()
def nli_matrix(sentences, samples, batch_size=20):
    # [n_sentences x n_samples] matrix of P(contradiction); batched version of SelfCheckNLI.predict
    M = np.zeros((len(sentences), len(samples)))
    for si, sentence in enumerate(sentences):
        for b in range(0, len(samples), batch_size):
            chunk = samples[b:b + batch_size]
            inputs = selfcheck_nli.tokenizer.batch_encode_plus(
                batch_text_or_text_pairs=[(sentence, s) for s in chunk],
                add_special_tokens=True, padding="longest", truncation=True,
                return_tensors="pt", return_token_type_ids=True, return_attention_mask=True,
            ).to(device)
            probs = torch.softmax(selfcheck_nli.model(**inputs).logits, dim=-1)
            M[si, b:b + len(chunk)] = probs[:, 1].cpu().numpy()      # class 1 = contradiction (neutral already removed)
    return M
ram("after loading NLI model")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

SelfCheck-NLI initialized to device cuda
[RAM after loading NLI model] this process 3.4 GB | system used 4.3 / 12.7 GB


In [ ]:
if VERIFY_AGAINST_PACKAGE:
    r = rows[IDS[0]]; sents, samp = r["gpt3_sentences"][:3], r["gpt3_text_samples"][:5]
    ref = selfcheck_nli.predict(sentences=sents, sampled_passages=samp)
    check_close("NLI batched re-implementation vs package", nli_matrix(sents, samp).mean(axis=1), ref, 1e-4)

t0 = time.time()
MATS["nli"]["samples"] = run_cached("nli_samples", lambda i: nli_matrix(rows[i]["gpt3_sentences"], get_samples(i)), IDS, "SelfCheckNLI", save_every=10)
print(f"done in {(time.time() - t0) / 60:.1f} min")
register_main("SelfCheck-NLI", scores_from_matrices("nli", MATS["nli"]["samples"]))

y, s = task_arrays(SENT_MAIN["SelfCheck-NLI"])["NonFact"]
print(f"NonFact: trapezoidal AUC-PR (paper protocol) = {100 * auc_pr(y, s):.2f}   |   average_precision_score = {100 * average_precision_score(y, s):.2f}")

# a few example sentences
r, M = rows[IDS[0]], MATS["nli"]["samples"][IDS[0]]
for sent, sc, lab in list(zip(r["gpt3_sentences"], M.mean(axis=1), r["annotation"]))[:5]:
    print(f"[{sc:.2f}] ({lab}) {sent[:90]}")

if RUN_WIKIBIO_EVIDENCE:
    MATS["nli"]["wiki"] = run_cached("nli_wiki", lambda i: nli_matrix(rows[i]["gpt3_sentences"], get_samples(i, "wiki")), IDS, "NLI (WikiBio)", save_every=20)
    RES_WIKI["WikiBio+NLI"] = evaluate(scores_from_matrices("nli", MATS["nli"]["wiki"]))


[OK] NLI batched re-implementation vs package: max |diff| = 2.03e-07 (tol 0.0001)
SelfCheckNLI: 0 cached, 238 to compute


SelfCheckNLI:   0%|          | 0/238 [00:00<?, ?it/s]

done in 38.8 min
SelfCheck-NLI {'NonFact': np.float64(92.5), 'NonFact*': np.float64(45.17), 'Factual': np.float64(66.08), 'Pearson': np.float64(74.14), 'Spearman': np.float64(73.78), 'n_passages': 238}
NonFact: trapezoidal AUC-PR (paper protocol) = 92.50   |   average_precision_score = 92.50
[0.87] (major_inaccurate) John Russell Reynolds (1820–1876) was an English lawyer, judge, and author.
[0.93] (major_inaccurate) He was born in London, the son of a barrister, and was educated at Eton College and Trinit
[0.93] (major_inaccurate) He was called to the bar in 1845, and became a Queen's Counsel in 1859.
[0.98] (major_inaccurate) He was appointed a judge of the Court of Common Pleas in 1867, and was knighted in 1871.
[0.22] (major_inaccurate) Reynolds was a prolific author, writing on a wide range of topics.
NLI (WikiBio): 0 cached, 238 to compute


NLI (WikiBio):   0%|          | 0/238 [00:00<?, ?it/s]